In [ ]:
import glob, os, subprocess, sys
hits = glob.glob('/kaggle/input/**/qwen38-wheelhouse-py312', recursive=True)
WH = hits[0] if hits else None
print('WHEELHOUSE:', WH, flush=True)
print('INPUT_ROOT:', sorted(glob.glob('/kaggle/input/*')), flush=True)
if WH:
    print(sorted(os.listdir(WH)), flush=True)
r0 = subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True, text=True)
print((r0.stdout or '')[-500:], flush=True)
if not WH:
    raise SystemExit('NO_WHEELHOUSE')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links=%s' % WH, 'transformers==5.8.0', 'accelerate', 'peft'], capture_output=True, text=True)
print((r.stdout or '')[-1500:], flush=True)
print((r.stderr or '')[-500:], flush=True)
if r.returncode != 0:
    raise SystemExit('WHEEL_INSTALL_FAILED %s' % r.returncode)
import transformers
print('TF_NOW:', transformers.__version__, flush=True)


In [ ]:
import glob, json, os, subprocess, sys
NEED = ('solve_grid.py', 'arc_episode_data.py', 'split.json', 'adapter_config.json', 'adapter_model.safetensors')
def find_code_dir():
    bases = [os.getcwd(), '/kaggle/working', '/kaggle/src', '/kaggle/input']
    found = []
    seen = set()
    for base in bases:
        if not base or not os.path.isdir(base):
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            dirnames[:] = [d for d in dirnames if d not in ('.git', '__pycache__')]
            if all(name in filenames for name in NEED):
                if dirpath not in seen:
                    seen.add(dirpath)
                    found.append(dirpath)
            depth = dirpath.count(os.sep) - base.count(os.sep)
            if depth >= 4:
                dirnames.clear()
    return found
code_dirs = find_code_dir()
print('CODE_DIRS:', code_dirs, flush=True)
if not code_dirs:
    raise SystemExit('CODE_NOT_FOUND')
code_dir = code_dirs[0]
print('CODE_DIR:', code_dir, flush=True)
print('FILES:', sorted(os.listdir(code_dir)), flush=True)
ch = glob.glob('/kaggle/input/**/arc-agi_training_challenges.json', recursive=True)
so = glob.glob('/kaggle/input/**/arc-agi_training_solutions.json', recursive=True)
print('CHALLENGES:', ch, flush=True)
print('SOLUTIONS:', so, flush=True)
if not ch or not so:
    raise SystemExit('COMPETITION_JSON_NOT_FOUND')
model_dir = None
for path in glob.glob('/kaggle/input/**/config.json', recursive=True):
    try:
        cfg = json.load(open(path, encoding='utf-8'))
    except Exception:
        continue
    if cfg.get('model_type') in ('qwen3_5', 'qwen3'):
        model_dir = os.path.dirname(path)
        break
print('MODEL_DIR:', model_dir, flush=True)
if not model_dir:
    raise SystemExit('MODEL_DIR_NOT_FOUND')
out_dir = '/kaggle/working/007-g1-eval'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'solve.json')
cmd = [
    sys.executable,
    os.path.join(code_dir, 'solve_grid.py'),
    '--model-dir', model_dir,
    '--adapter', code_dir,
    '--challenges', ch[0],
    '--solutions', so[0],
    '--split-file', os.path.join(code_dir, 'split.json'),
    '--out', out_path,
    '--max-new-tokens', '960',
    '--time-budget', '2000',
    '--n-sample', '1',
    '--seed', '20260930',
]
print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd)
print('SOLVE_EXIT', proc.returncode, flush=True)
print('SOLVE_EXISTS', os.path.isfile(out_path), flush=True)
if os.path.isfile(out_path):
    payload = json.load(open(out_path, encoding='utf-8'))
    aggregates = payload.get('aggregates') if isinstance(payload, dict) else None
    print('SOLVE_JSON_KEYS', sorted(payload)[:20] if isinstance(payload, dict) else type(payload).__name__, flush=True)
    print('TIME_GUARD', payload.get('time_guard') if isinstance(payload, dict) else None, flush=True)
    print('N_QUERIES', len(payload.get('queries') or []) if isinstance(payload, dict) else None, flush=True)
    if aggregates is not None:
        print(json.dumps(aggregates, ensure_ascii=False)[:4000], flush=True)
if proc.returncode not in (0, 2):
    raise SystemExit(proc.returncode)
